<a href="https://colab.research.google.com/github/OlaSletten/AppsTypeMarks/blob/main/quickstarts/keras_quickstart.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Keras quickstart

We recommend running this example in Colab's GPU runtime. It will run on Jax, TensorFlow or PyTorch, simply change the line below.

In [ ]:
import os
os.environ["KERAS_BACKEND"] = "jax"  # Or "torch" or "tensorflow"

In [ ]:
# Install Keras and JAX backend dependencies when running in a fresh environment.
%pip install -q -U keras jax jaxlib

## Train an MNIST classifier with a mini ResNet model

In [ ]:
import keras
from keras.datasets import mnist
from keras import layers

In [ ]:
(x_train, y_train), (x_test, y_test) = mnist.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0
# Conv2D expects inputs as (batch, height, width, channels).
x_train = x_train[..., None]
x_test = x_test[..., None]

inputs = keras.Input(shape=(28, 28, 1))
x = layers.Conv2D(32, 3, activation="relu")(inputs)
x = layers.Conv2D(64, 3, activation="relu")(x)
residual = x = layers.MaxPooling2D(3)(x)

x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = layers.Conv2D(64, 3, activation="relu", padding="same")(x)
x = x + residual

x = layers.Conv2D(64, 3, activation="relu")(x)
x = layers.GlobalAveragePooling2D()(x)
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.5)(x)
outputs = layers.Dense(10, activation="softmax")(x)

model = keras.Model(inputs, outputs, name="mini_resnet")

In [ ]:
keras.utils.plot_model(model, "mini_resnet.png", dpi=100, show_shapes=True)

In [ ]:
model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)
model.fit(x_train, y_train, epochs=20)
model.evaluate(x_test, y_test)